In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_9.pickle

trying: ['file_name_2018', 'file_name_2017']
me:  2
me:  2
trying: ['file_name_2018', 'file_name_2017']
me:  2
me:  2
trying: ['responses_df_2022_cell10', 'responses_df_2022']


me:  10
me:  10
trying: ['responses_df_2022_cell10', 'responses_df_2022']


me:  10
me:  10
trying: ['title_for_x_axis', 'title_for_x_axis_cell20']
me:  12
me:  16
trying: ['title_for_x_axis', 'title_for_x_axis_cell20']
me:  12
me:  16
trying: ['question_name', 'question_of_interest_cell18']
me:  12
me:  12
trying: ['question_name', 'question_of_interest_cell18']
me:  12
me:  12
trying: ['sns']
me:  0
trying: ['np']
me:  0
trying: ['add_year_column_to_dataframes_18']
me:  12
trying: ['warnings']
me:  0
trying: ['go']
me:  0
trying: ['glob']
me:  0
trying: ['pd']
me:  0
trying: ['replace_hyphen_with_en_dash']
me:  4
trying: ['percentages_per_country_df']
me:  8
trying: ['create_dataframe_of_counts_16']
me:  8
trying: ['count_then_return_percent_21']
me:  18
trying: ['responses_df_2018']


me:  2
trying: ['responses_in_order_cell21']
me:  18
trying: ['base_dir_2022']
me:  2
trying: ['base_dir_2017']
me:  2
trying: ['responses_df_2019']


me:  2
trying: ['question_of_interest_cell19']
me:  14
trying: ['file_name_2020']
me:  2
trying: ['count_then_return_percent_19']
me:  14
trying: ['percentages_cell19']
me:  14
trying: ['base_dir_2018']
me:  2
trying: ['file_name_2022']
me:  2
trying: ['percentages']
me:  10
trying: ['file_name_2021']
me:  2
trying: ['directory']
me:  2
trying: ['responses_df_2020']


...

me:  12
trying: ['file_name_2019']
me:  2
trying: ['responses_df_2021']


me:  12
trying: ['percentages_cell17']
me:  10
trying: ['load_survey_data']
me:  2
trying: ['responses_df_2019_cell10']


me:  12
trying: ['count_then_return_percent_17']
me:  10
trying: ['base_dir_2020']
me:  2
trying: ['responses_in_order']
me:  10
trying: ['directory_cell8']
me:  2
trying: ['responses_df_2018_cell10']


me:  16
trying: ['orientation_for_chart']
me:  10
trying: ['base_dir_2019']
me:  2
trying: ['question_of_interest_cell20']
me:  16
trying: ['question_of_interest']
me:  10
trying: ['factor']
me:  2
trying: ['combine_subset_of_data_from_multiple_years_20']
me:  16
trying: ['px']
me:  0
trying: ['question_name_alternate_cell18']
me:  12
trying: ['count_then_return_percent_20']
me:  16
trying: ['orig_output']
me:  1
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  12
trying: ['add_year_column_to_dataframes_20']
me:  16
trying: ['percentages_cell21']
me:  18
trying: ['age_df_combined']
me:  16
trying: ['question_of_interest_cell21']
me:  18
trying: ['country_df_combined_cell18']
me:  12
trying: ['count_then_return_percent_18']
me:  12
trying: ['country_df_combined']
me:  12
trying: ['question_name_alternate']
me:  12
trying: ['subset_of_countries']
me:  12


Declaring variable sns
Declaring variable np
Declaring variable warnings
Declaring variable go
Declaring variable glob
Declaring variable pd
Declaring variable px
Declaring variable orig_output
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable responses_df_2018
Declaring variable base_dir_2022
Declaring variable base_dir_2017
Declaring variable responses_df_2019
Declaring variable file_name_2020
Declaring variable base_dir_2018
Declaring variable file_name_2022
Declaring variable file_name_2021
Declaring variable directory
Declaring variable base_dir_2021
Declaring variable file_name_2019
Declaring variable load_survey_data
Declaring variable base_dir_2020
Declaring variable directory_cell8
Declaring variable base_dir_2019
Declaring variable factor
Declaring variable replace_hyphen_with_en_dash
Declaring variable percentages_per_country_df
Declaring variable create_dataframe_of_coun

In [3]:
%%RecordEvent
%%time
### cell 10 ###

gender_map = {
    'Male': 'Man',
    'Female': 'Woman',
    'A different identity': 'Prefer to self-describe',
    'Non-binary, genderqueer, or gender non-conforming': 'Prefer to self-describe',
    'Nonbinary': 'Prefer to self-describe',
    'Prefer not to say': 'Prefer to self-describe'
}


def combine_subset_of_data_from_multiple_years_22(question_of_interest, x_axis_title, include_2017=None):
    # Prepare list of (year, df, column)
    year_dfs = []
    if include_2017 is not None:
        year_dfs.append(('2017', responses_df_2017, 'GenderSelect'))
    year_dfs += [
        ('2022', responses_df_2022_cell10, question_of_interest),
        ('2021', responses_df_2021, question_of_interest),
        ('2020', responses_df_2020, question_of_interest),
        ('2019', responses_df_2019_cell10, question_of_interest),
        ('2018', responses_df_2018_cell10, question_of_interest),
    ]

    frames = []
    for year, df, col in year_dfs:
        # Fast count by original category, then map and aggregate counts only on unique keys
        counts = df[col].value_counts(dropna=False)
        # Consolidate categories via the gender_map on the counts index
        grouped = counts.groupby(
            counts.index.to_series().replace(gender_map),
            dropna=False
        ).sum()
        # Compute percentages
        pct = (grouped.div(grouped.sum()).mul(100).round(1))
        # Build a small DataFrame from the percentages
        tmp = pct.reset_index(name='percentage')
        # Rename the first column (the category) to the x-axis title
        tmp.columns = [x_axis_title, 'percentage']
        tmp['year'] = year
        frames.append(tmp)

    # Combine all years and ensure column order
    result = pd.concat(frames, ignore_index=True)
    return result[['percentage', 'year', x_axis_title]]

# Combine and sort
question_of_interest_cell22 = 'What is your gender? - Selected Choice'
title_for_x_axis_cell22 = ''
age_df_combined_cell22 = (
    combine_subset_of_data_from_multiple_years_22(
        question_of_interest_cell22,
        title_for_x_axis_cell22,
        include_2017='yes'
    )
    .sort_values(['year', 'percentage'], ascending=True)
)
age_df_combined_cell22.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19 entries, 3 to 4
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   percentage  19 non-null     float64
 1   year        19 non-null     object 
 2               18 non-null     object 
dtypes: float64(1), object(2)
memory usage: 608.0+ bytes
CPU times: user 23.1 ms, sys: 0 ns, total: 23.1 ms
Wall time: 23.1 ms


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_10_try_1.pickle

migration speed (bps): 748508976.8400236


---------------------------
variables to migrate:
responses_df_2020 255836653
replace_hyphen_with_en_dash 144
create_dataframe_of_counts_16 144
responses_df_2017 157241558
title_for_x_axis_cell22 49
percentages_per_country_df 4474
responses_df_2021 342786145
responses_df_2019_cell10 160741165
question_of_interest_cell22 87
title_for_x_axis_cell20 49
age_df_combined_cell22 2723
count_then_return_percent_20 144
base_dir_2022 155
add_year_column_to_dataframes_20 144
base_dir_2017 155
age_df_combined 10611
file_name_2020 81
responses_df_2018_cell10 320850950
count_then_return_percent_19 144
file_name_2018 76
question_of_interest_cell20 76
percentages_cell19 784
base_dir_2018 155
combine_subset_of_data_from_multiple_years_20 144
question_of_interest_cell19 76
file_name_2017 76
file_name_2022 81
file_name_2021 81
directory 163
base_dir_2021 155
file_name_2019 78
percentages_cell17 958
load_survey_data 144
count_then_return_percent_17 144
sns 72
base_dir_2020 155
responses_in_order 184
np 72


In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['responses_df_2020', 'responses_df_2019', 'responses_df_2018', 'responses_df_2022', 'responses_df_2021']
Intermediate variables ['file_name_2022', 'base_dir_2020', 'file_name_2018', 'directory', 'file_name_2020', 'base_dir_2022', 'directory_cell8', 'file_name_2017', 'base_dir_2021', 'factor', 'responses_df_2017', 'file_name_2019', 'file_name_2021', 'base_dir_2019', 'base_dir_2017', 'base_dir_2018']
Future variables ['percentages', 'responses_df_2017', 'responses_df_2022_cell10']
Modified dataframes
======= Cell 2 =======
Input variables ['responses_df_2019', 'responses_df_2018', 'responses_df_2022']
Active variables ['responses_df_2022', 'responses_df_2022_cell10', 'responses_df_2019_cell10']
Intermediate variables ['responses_df_2018_cell10']
Future variables ['responses_df_2022_cell10', 'percentages

In [6]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_10_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[10], f)


In [7]:
opt_output = Out.get(4)

In [8]:
responses_df_2022_cell10_opt = responses_df_2022_cell10
title_for_x_axis_cell22_opt = title_for_x_axis_cell22
responses_df_2021_opt = responses_df_2021
responses_df_2022_opt = responses_df_2022
responses_df_2020_opt = responses_df_2020
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_10.pickle
assert title_for_x_axis_cell22_opt == title_for_x_axis_cell22

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['title_for_x_axis_cell22', 'title_for_x_axis', 'title_for_x_axis_cell20']
me:  20
me:  12
me:  16
trying: ['title_for_x_axis_cell22', 'title_for_x_axis', 'title_for_x_axis_cell20']
me:  20
me:  12
me:  16
trying: ['file_name_2017', 'file_name_2018']
me:  2
me:  2
trying: ['file_name_2017', 'file_name_2018']
me:  2
me:  2
trying: ['responses_df_2022_cell10', 'responses_df_2022']


me:  20
me:  20
trying: ['responses_df_2022_cell10', 'responses_df_2022']


me:  20
me:  20
trying: ['question_name', 'question_of_interest_cell18']
me:  12
me:  12
trying: ['question_name', 'question_of_interest_cell18']
me:  12
me:  12
trying: ['responses_df_2018']


me:  2
trying: ['question_of_interest_cell21']
me:  18
trying: ['percentages']
me:  10
trying: ['count_then_return_percent_21']
me:  18
trying: ['responses_df_2021']


me:  20
trying: ['percentages_cell21']
me:  18
trying: ['responses_in_order_cell21']
me:  18
trying: ['responses_df_2019']


...

me:  20
trying: ['file_name_2022']
me:  2
trying: ['file_name_2021']
me:  2
trying: ['directory']
me:  2
trying: ['base_dir_2021']
me:  2
trying: ['question_of_interest_cell22']
me:  20
trying: ['file_name_2019']
me:  2
trying: ['percentages_cell17']
me:  10
trying: ['load_survey_data']
me:  2
trying: ['count_then_return_percent_17']
me:  10
trying: ['sns']
me:  0
trying: ['base_dir_2020']
me:  2
trying: ['responses_in_order']
me:  10
trying: ['np']
me:  0
trying: ['add_year_column_to_dataframes_22']
me:  20
trying: ['orientation_for_chart']
me:  10
trying: ['warnings']
me:  0
trying: ['directory_cell8']
me:  2
trying: ['question_of_interest']
me:  10
trying: ['go']
me:  0
trying: ['px']
me:  0
trying: ['base_dir_2019']
me:  2
trying: ['glob']
me:  0
trying: ['responses_df_2018_cell10']


me:  20
trying: ['factor']
me:  2
trying: ['combine_subset_of_data_from_multiple_years_22']
me:  20
trying: ['orig_output']
me:  1
trying: ['responses_df_2020']


me:  20
trying: ['pd']
me:  0
trying: ['question_name_alternate_cell18']
me:  12
trying: ['add_year_column_to_dataframes_18']
me:  12
trying: ['age_df_combined_cell22']
me:  20
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  12
trying: ['title_for_x_axis']
me:  12
trying: ['country_df_combined_cell18']
me:  12
trying: ['count_then_return_percent_18']
me:  12
trying: ['country_df_combined']
me:  12
trying: ['count_then_return_percent_22']
me:  20
trying: ['question_name_alternate']
me:  12
trying: ['subset_of_countries']
me:  12


Declaring variable sns
Declaring variable np
Declaring variable warnings
Declaring variable go
Declaring variable px
Declaring variable glob
Declaring variable pd
Declaring variable orig_output
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable responses_df_2018
Declaring variable responses_df_2019
Declaring variable base_dir_2022
Declaring variable base_dir_2017
Declaring variable file_name_2020
Declaring variable base_dir_2018
Declaring variable file_name_2022
Declaring variable file_name_2021
Declaring variable directory
Declaring variable base_dir_2021
Declaring variable file_name_2019
Declaring variable load_survey_data
Declaring variable base_dir_2020
Declaring variable directory_cell8
Declaring variable base_dir_2019
Declaring variable factor
Declaring variable replace_hyphen_with_en_dash
Declaring variable create_dataframe_of_counts_16
Declaring variable percentages_per_count

Declaring variable combine_subset_of_data_from_multiple_years_22
Declaring variable responses_df_2020
Declaring variable age_df_combined_cell22
Declaring variable count_then_return_percent_22
